# 073 — Tokenización moderna y vocabularios

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** Corpus inicial: `a b _ ×4`, `a b c _ ×3`, `b c d _ ×2`.
Pares: (a,b)=7, (b,_)=4, (b,c)=5, (c,_)=3, (c,d)=2, (d,_)=2.
Merge 1: (a,b)→`ab` (7). Corpus: `ab _`, `ab c _`, `b c d _`.
Nuevos pares: (ab,_)=4, (ab,c)=3, (b,c)=2, (c,_)=3, (c,d)=2, (d,_)=2.
Merge 2: (ab,_)→`ab_` (4). Merge 3: (ab,c)→`abc` o (c,_)→`c_`, ambos con
frecuencia 3 — el desempate depende de la convención (orden de aparición); con
desempate por aparición gana (ab,c)→`abc`.

**Ejercicio 2.** `w e s t _` → (e,s)→`es`: `w es t _` → (es,t)→`est`: `w est _`
→ (est,_)→`est_`: `w est_`. Las merges `lo`/`low` no aplican.
Resultado: **2 tokens**, `w` + `est_`.

**Ejercicio 3.** Fertilidad EN = 1200/900 ≈ 1,33 tokens/palabra;
ES = 1700/950 ≈ 1,79. Sobrecosto por token: 1700/1200 − 1 ≈ **41,7 %** para un
contenido comparable.

**Ejercicio 4.** El código valida el contrato; la semilla fija la parte
pseudoaleatoria del experimento, por eso la evidencia es reproducible.

In [ ]:
# Ejercicio 2 verificado en código
merges = [("e", "s"), ("es", "t"), ("est", "_"), ("l", "o"), ("lo", "w")]
word = ["w", "e", "s", "t", "_"]
for a, b in merges:
    i = 0
    while i < len(word) - 1:
        if word[i] == a and word[i + 1] == b:
            word[i:i + 2] = [a + b]
        else:
            i += 1
print("west ->", word)  # ['w', 'est_']

# Ejercicio 3
fert_en, fert_es = 1200 / 900, 1700 / 950
print(f"fertilidad EN={fert_en:.2f} ES={fert_es:.2f} sobrecosto={(1700/1200-1):.1%}")

# Ejercicio 4
result = run_lab("llm", seed=73)
assert result["kind"] == "llm"
assert result["evidence"] and result["limitations"]
show(result)

## Reflexión

1. ¿Por qué un vocabulario BPE entrenado sobre texto mayoritariamente inglés castiga
   (en tokens y en dinero) a quien escribe en guaraní o quechua?
2. Si "1234" es un token único pero "1235" son dos, ¿qué implica para pedirle
   aritmética a un LLM y cómo lo mitigarías?
3. ¿Qué ventaja concreta aporta el muestreo de segmentaciones del modelo unigram
   durante el entrenamiento que BPE determinista no ofrece?